# 01 · Exporturi Earth Engine (pentru QGIS și vizualuri)
Exporturile rulează pe serverele Earth Engine și ajung în Google Drive, folderul `GISDay2026`.
Nu au nevoie de `drive.mount()`. Progresul: https://code.earthengine.google.com/tasks

Statisticile pe parcelă NU depind de aceste fișiere: notebook-ul 03 le calculează direct în Earth Engine.

In [ ]:
# Sincronizare: aduce repo-ul pe runtime și încarcă configurarea.
# Local (kernel pe PC) găsește direct ../gisday_config.py; pe Colab clonează repo-ul privat.
import os, sys, subprocess, base64, getpass

def _auth(tok):
    b = base64.b64encode(f'x-access-token:{tok}'.encode()).decode()
    return ['-c', f'http.extraheader=AUTHORIZATION: basic {b}']

def _token():
    global TOKEN
    if 'TOKEN' not in globals():
        TOKEN = getpass.getpass('Token GitHub (fine-grained, doar repo-ul atlas-seceta-2024): ')
    return TOKEN

if not os.path.exists('gisday_config.py'):
    if os.path.exists('../gisday_config.py'):
        os.chdir('..')
    else:
        REPO = 'https://github.com/lefterpatrickandrei-sketch/atlas-seceta-2024.git'   # același cu GITHUB_REPO din gisday_config.py
        if not os.path.exists('/content/atlas-seceta-2024'):
            subprocess.run(['git', *_auth(_token()), 'clone', REPO, '/content/atlas-seceta-2024'], check=True)
        os.chdir('/content/atlas-seceta-2024')
sys.path.insert(0, os.getcwd())
from gisday_config import *

def pull():
    subprocess.run(['git', *_auth(_token()), 'pull', '--ff-only'], check=True)

def push(paths, msg):
    subprocess.run(['git', 'add', *[str(p) for p in paths]], check=True)
    subprocess.run(['git', '-c', 'user.name=gisday-runtime', '-c', 'user.email=gisday@users.noreply.github.com',
                    'commit', '-m', msg], check=False)
    subprocess.run(['git', *_auth(_token()), 'push'], check=True)

print('Lucrez în', os.getcwd())

In [ ]:
import ee
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode='notebook')   # afișează un link; lipești codul primit
    ee.Initialize(project=GEE_PROJECT)
BBOX = ee.Geometry.Rectangle(list(AOI_BBOX))
RO = ee.FeatureCollection('FAO/GAUL/2015/level0').filter(ee.Filter.eq('ADM0_NAME', 'Romania'))
AOI = BBOX.intersection(RO.geometry(), ee.ErrorMargin(10))
print('Earth Engine OK')

In [ ]:
S2_BANDS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']

def s2(start, end):
    def mask(img):
        scl = img.select('SCL')
        bad = scl.eq(3).Or(scl.eq(8)).Or(scl.eq(9)).Or(scl.eq(10)).Or(scl.eq(11))
        return img.updateMask(bad.Not()).select(S2_BANDS).divide(10000)
    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(AOI).filterDate(start, end)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60)).map(mask))

S1 = (ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(AOI)
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
      .select(['VV', 'VH']))

In [ ]:
FOLDER = 'GISDay2026'

def export_img(img, name, scale):
    t = ee.batch.Export.image.toDrive(image=img.clip(AOI), description=name, folder=FOLDER, fileNamePrefix=name,
                                      region=AOI, scale=scale, crs=CRS, maxPixels=1e10)
    t.start(); print('pornit:', name)

def export_table(fc, name):
    t = ee.batch.Export.table.toDrive(collection=fc, description=name, folder=FOLDER, fileNamePrefix=name,
                                      fileFormat='CSV')
    t.start(); print('pornit:', name)

def indices(comp):
    return (comp.normalizedDifference(['B8', 'B4']).rename('NDVI')
            .addBands(comp.normalizedDifference(['B8', 'B11']).rename('NDMI')).multiply(10000).toInt16())

In [ ]:
# NDVI + NDMI lunar, aprilie–octombrie
for m in range(4, 11):
    st = ee.Date.fromYMD(YEAR, m, 1)
    export_img(indices(s2(st, st.advance(1, 'month')).median()), f'S2_idx_{YEAR}_{m:02d}', 10)
# reflectanță pentru vizualuri: mai și august
for m in (5, 8):
    st = ee.Date.fromYMD(YEAR, m, 1)
    export_img(s2(st, st.advance(1, 'month')).median().multiply(10000).toInt16(), f'S2_refl_{YEAR}_{m:02d}', 10)
# NDVI iulie–august, 2017–2025
for y in range(2017, 2026):
    export_img(indices(s2(ee.Date.fromYMD(y, 7, 1), ee.Date.fromYMD(y, 9, 1)).median()).select('NDVI'),
               f'S2_ndvi_JA_{y}', 10)
# Sentinel-1 lunar (dB x100)
for m in range(4, 11):
    st = ee.Date.fromYMD(YEAR, m, 1)
    export_img(S1.filterDate(st, st.advance(1, 'month')).mean().multiply(100).toInt16(), f'S1_dB_{YEAR}_{m:02d}', 10)

In [ ]:
DEM_IC = ee.ImageCollection('COPERNICUS/DEM/GLO30').select('DEM')
export_img(DEM_IC.mosaic().toFloat(), 'DEM_GLO30', 30)
export_img(ee.ImageCollection('JRC/D5/EUCROPMAP/V1').filterDate('2022-01-01', '2023-01-01').first().toInt16(),
           'EUCROPMAP_2022', 10)

In [ ]:
# ERA5-Land orar în centrul zonei (roza vânturilor) și zilnic pe zonă (anomalii)
center = AOI.centroid(100)
hourly = (ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY').filterDate(f'{YEAR}-05-01', f'{YEAR}-10-01')
          .select(['temperature_2m', 'u_component_of_wind_10m', 'v_component_of_wind_10m', 'total_precipitation']))
export_table(hourly.map(lambda i: ee.Feature(None, i.reduceRegion(ee.Reducer.first(), center, 11000))
                        .set('time', i.date().format('YYYY-MM-dd HH:mm'))), f'ERA5L_hourly_center_{YEAR}')
daily = (ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR').filterDate('2017-01-01', '2026-01-01')
         .select(['temperature_2m_max', 'total_precipitation_sum']))
export_table(daily.map(lambda i: ee.Feature(None, i.reduceRegion(ee.Reducer.mean(), AOI, 11000))
                       .set('date', i.date().format('YYYY-MM-dd'))), 'ERA5L_daily_aoi_2017_2025')